# Learning about Pre-trained Transformers and how to use them

## From Raw Language Models to ChatGPT

### Model Scaling and Capabilities

One clear trend in this field: the bigger the model, the more capable it usually becomes, and there are already models with upwards of a trillion parameters. However, scaling alone isn't enough to produce something like ChatGPT. A model trained purely with next-token prediction (exactly like the `DemoGPT` we built) learns to generate *likely continuations* of text — nothing more. Depending on model size, training data, and how it's prompted, such a model can stumble into answering questions correctly some of the time, but it has no built-in notion of "the user asked a question, so I should answer it directly." Reliable, instruction-following behavior requires additional training stages on top of raw next-token prediction.

### Example of Model Limitations

An older version of GPT-3 — trained purely to generate text, not to answer questions — illustrates this well. Given a factual question as a prompt, it doesn't answer it; it continues the text in whatever way looks statistically likely to follow that kind of sentence (e.g., generating more questions in the same style, rephrasing it, or drifting into a related topic). It was trained on a large corpus of text that *surrounds* text like the prompt, not on text that *answers* the prompt. Send the same question to ChatGPT, though, and it reliably returns a direct answer (e.g., "Paris"), often with added context — because ChatGPT went through extra training stages specifically designed to produce that behavior.

### How ChatGPT Was Trained

At a high level, OpenAI's process for turning a raw GPT model into something like ChatGPT involves three stages:

1. **Pretraining**: Train a GPT base model similar in architecture to the one built in this notebook series, but far larger and on far more data. This is the next-token-prediction stage, and it's where most of the model's raw language and world knowledge comes from.
2. **Supervised Fine-Tuning (SFT)**: Collect prompts paired with high-quality, human-written example responses (question-answer pairs and other instruction-following tasks), then fine-tune the base model on this dataset so it starts producing direct, helpful responses instead of generic continuations.
3. **Reinforcement Learning from Human Feedback (RLHF)**: Further refine the SFT model's responses to better match human preferences, using a *reward model* trained on human rankings (details below).

This entire process — taking a model trained for one task and adjusting its weights so it specializes in a different, related task — is called **fine-tuning**, and it's a general deep learning technique used well beyond LLMs (e.g., fine-tuning an image classifier pretrained on ImageNet to recognize a new, narrower set of classes). It's worth noting that "GPT-3.5" refers to an entire *family* of models produced through this pipeline, not the output of a single fine-tuning step.

### The Reward Model and RLHF, in More Detail

The RLHF stage is the part of this pipeline that's easiest to gloss over, but it's worth unpacking since it's what actually teaches the model to prefer *good* answers over merely *plausible-looking* ones:

1. **Sampling multiple answers**: For a given prompt, the SFT model generates several different candidate answers (since sampling is stochastic — recall the `torch.multinomial` call in our own `generate` function — the same prompt can produce different outputs each time).
2. **Human ranking**: People rank these candidate answers from best to worst, based on helpfulness, correctness, and safety. This produces a dataset of *relative preferences* (e.g., "answer B is better than answer A") rather than a single "correct" answer — which is a much easier and more scalable thing to ask humans to do than writing ideal answers from scratch for every possible prompt.
3. **Training the reward model**: This preference dataset is used to train a separate model — the **reward model** — whose job is to take a (prompt, answer) pair and output a single scalar score estimating how good a human would rate that answer. Once trained, this model can score *any* answer automatically, without needing a human in the loop for every single comparison.
4. **Optimizing the language model with RL**: Finally, the SFT model is further tuned using reinforcement learning (commonly Proximal Policy Optimization, or PPO), where the reward model's score acts as the reward signal — the language model is nudged to generate text that the reward model scores highly, while a penalty term keeps it from straying too far from its original SFT behavior (to avoid it gaming the reward model with degenerate but high-scoring text).

**Real-life analogy**: imagine training a new employee. Pretraining is like them reading everything in the company library — they absorb a huge amount of general knowledge but don't yet know how to actually do the job. Supervised fine-tuning is like shadowing a few expert employees and copying exactly what they do in specific situations. RLHF is like having a manager repeatedly compare pairs of the new employee's work and say "this one's better," until the employee develops a general sense for what good work looks like — even in situations they weren't explicitly shown examples for.

The datasets used for both the SFT and reward-model stages are proprietary to OpenAI and are considerably harder (and more expensive) to collect than the raw internet text used for pretraining, since they require paid human annotators rather than just scraping publicly available text.

### Comparing Models: Our Model vs. OpenAI's GPT Series

| Model | Parameters | Layers | Training Data |
|---|---|---|---|
| **Our `DemoGPT`** | ~85 million (with `embedding_dim=768`, `layers_num=10`) | 10 | ~1 MB (Tiny Shakespeare) |
| **GPT-1** | 120 million | 12 | 4.5 GB |
| **GPT-2** | ~10x GPT-1 | ~4x GPT-1 | ~10x GPT-1 |
| **GPT-3** | Significantly more than GPT-2 (175 billion, in the largest published variant) | More than GPT-2 | Substantially larger than GPT-2 |
| **Recent OpenAI models** | Rumored to exceed GPT-3's parameter count | — | — |

It's striking just how structurally similar our `DemoGPT` is to GPT-1 — both are decoder-only Transformers built from token embeddings, positional embeddings, stacked masked multi-head attention + feedforward blocks, and a final linear "unembedding" layer. The gap between our toy model and something like GPT-3 isn't really architectural; it's almost entirely scale (parameters, layers, context size) and training data volume, plus the SFT/RLHF stages layered on top.

### A Note on "Mixture of Experts"

Recent OpenAI models are rumored to use a **mixture-of-experts (MoE)** architecture rather than one single dense Transformer stack. The core idea: instead of every token passing through the *same* feedforward block in every layer (as in our `FeedForward` class), an MoE model has several parallel feedforward "expert" sub-networks per layer, plus a small router network that decides which expert(s) should process each token. Only a subset of experts actually run for any given token, which means the *total* parameter count can be enormous while the *compute cost per token* stays much lower than running a single dense model of that same total size — a way to scale capacity without scaling inference cost proportionally.

### The Relationship Between Model Size and Capability — Key Takeaways

- Larger models tend to be more capable, with some now reaching into the trillion-parameter range.
- Scaling alone isn't sufficient: a large model trained only with next-token prediction still won't reliably behave like an instruction-following assistant.
- Turning a raw language model into a product like ChatGPT requires additional, deliberately designed training stages (SFT, then RLHF) layered on top of pretraining — and those stages depend on carefully curated, human-annotated datasets that are far more costly to produce than pretraining data.


## The Original Transformer: Encoder-Decoder Architecture

### Origin of Transformers

Transformers were introduced in the 2017 paper *"Attention Is All You Need."* That paper's architecture is more complex than the decoder-only model we built, and it was designed for a different purpose: not generating open-ended text, but **translating** text from one language to another (the paper's own experiments were on English-to-German and English-to-French translation).

### Key Components of Transformer Architecture

- **Decoder Component**: The right half of the original architecture, structurally similar to our model:
  - Embedding layer
  - Positional encoding
  - Masked multi-head attention (causal, using the same triangular mask we implemented)
  - Feedforward layer with residual connections
  - **Nx Repeated Blocks**: this block is stacked multiple times, just like our `layers_num` Transformer blocks.
- **Encoder Component**: The left half, absent from our model entirely. It encodes the *input* text (e.g., the English sentence to be translated) using multi-head attention **without** a causal mask, since the encoder is allowed to look at the entire input sentence at once — there's no "future" to hide, because it isn't generating anything token by token.
- **Interaction Between Encoder and Decoder**: This is the piece our simplified model doesn't have at all:
  - The encoder processes the full input sentence and produces a final-layer output representing it.
  - Each decoder block doesn't only attend to its own previous outputs — it also performs **cross-attention** against the encoder's output, which is how the decoder stays grounded in what the input sentence actually said while generating the translation.

### Cross-Attention: What Actually Changes

It's worth being explicit about what's different here from the masked self-attention we already built, since the terminology can blur together:

- In our `AttentionHead`, **Q, K, and V all come from the same source** (the decoder's own sequence) — this is *self-attention*.
- In the encoder-decoder's **cross-attention** layers, the **Q comes from the decoder** (the partially generated translation so far), while **K and V come from the encoder's output** (the fully encoded input sentence). The formula is identical — $\text{softmax}(QK^T / \sqrt{d_k})V$ — only the *source* of the matrices changes.
- This means every decoder token gets to directly "ask a question" of the entire input sentence at every layer, rather than needing that information to be squeezed entirely into the decoder's own token/positional embeddings.

**Real-life analogy**: imagine translating a sentence live, one word at a time, while occasionally glancing back at the original sentence to check you haven't drifted from its meaning. Self-attention is reviewing what you've translated *so far*; cross-attention is that glance back at the *original* text to stay faithful to it.

### Example: Translation from English to French

1. **Tokenization**: The English text is tokenized and converted into token IDs.
2. **Encoding**: Token IDs are passed through the encoder to build a representation of the phrase.
3. **Decoding**:
   - The process starts with a "start of sequence" token sent to the decoder.
   - The decoder reads its own input so far, plus the encoder's output (via cross-attention), to produce the next token ID of the translated phrase.
   - This repeats, generating one token at a time, until an "end of sequence" token is produced.
   - The final token IDs are converted back into a human-readable sentence.

Note how this generation loop is conceptually identical to our own `generate` function — autoregressive, one token at a time, stopping on a special token (in our case, when `max_tokens` runs out or `context_size` is reached rather than an explicit end-of-sequence token) — the encoder/cross-attention piece is additional context feeding into each step, not a replacement for that loop.

### Different Network Architectures Using Encoders and Decoders

| Architecture | Structure | Best suited for | Examples |
|---|---|---|---|
| **Decoder-only** | Decoder blocks only (masked self-attention, no cross-attention) — what we built | Open-ended text generation | GPT, LLaMA |
| **Encoder-decoder** | Both encoder and decoder blocks, connected via cross-attention | Sequence-to-sequence tasks like translation or summarization | T5, BART |
| **Encoder-only** | Encoder blocks only, no decoder at all | Understanding/classifying existing text (not generating new text) | BERT |

### Why the Split Exists

The common thread across all three: whether a model needs an encoder, a decoder, or both comes down to whether it needs to **generate a new sequence token-by-token** (decoder, with its causal mask) versus **build a full-context understanding of an existing sequence** (encoder, with unmasked attention) versus **both** (translation, summarization, and other sequence-to-sequence tasks). Decoder-only models like the one we built are the simplest of the three precisely because text generation alone only ever needs the "generate the next token given what came before" capability — there's no separate input sequence that needs to be encoded first.



## Using Pre-Trained Models Instead of Training From Scratch

While it is important to understand how large language models work under the hood, it is unlikely you will need to create one from scratch for your applications. Training an LLM from scratch is challenging for several reasons:

- **Data and Resources**: Training these models requires huge amounts of training data and a lot of computational resources.
- **Cost**: These models are very expensive to train. For example, the cost of training the GPT-4 model was more than $100 million.
- **Expertise**: Developing new models requires a specialized skill set, especially if you aim to implement a state-of-the-art model.

### Why This Is Worth Dwelling On

It's worth connecting this back to what we actually built in the companion notebook: our `DemoGPT` trained on roughly a megabyte of Shakespeare text, with 10 layers and a context size of 256, on a single machine. GPT-4-scale training involves datasets measured in terabytes, clusters of thousands of GPUs running for months, and teams of specialized researchers tuning the architecture, optimizer schedule, and data pipeline. The *mechanics* we implemented (embeddings, attention, feedforward blocks, cross-entropy loss, backprop) really are the same mechanics used at that scale — but the gap in engineering effort and cost is enormous, which is exactly why reusing an already-trained model is almost always the right call for building applications.

### Hugging Face: A Resource for Pre-Trained Models

Fortunately, instead of creating a model from scratch, we can use existing models created by companies and researchers from all around the world. To do this, we will use the [Hugging Face](https://huggingface.co/) library, which is one of the most popular NLP libraries for Python, with a large and active open-source community contributing new models and features.

Hugging Face simplifies building applications using LLMs but is not limited to this — it also provides tools for developing new LLMs. Hugging Face is a collection of multiple components that work together:

- **An API** for building applications using LLMs.
- **A large collection of pre-trained models** we can use directly in our applications.
- **Datasets** for different purposes, usable for training our own models or improving existing ones.
- **Spaces**, which let us quickly create and share machine learning demos and applications.

### API and Capabilities

Hugging Face provides a set of flexible and powerful APIs that we can use to build our AI applications:

- **Access existing models** for a variety of tasks, such as text generation and text summarization.
- **Train new models or fine-tune existing ones** from their library of models. Fine-tuning — a concept we already touched on with ChatGPT's training pipeline — takes an existing model and updates its weights to specialize it for a task it wasn't originally trained for.
- **Process text for LLMs**, for example by tokenizing text using an existing tokenizer or building a custom one (conceptually similar to the `CharTokenizer` we wrote ourselves, just typically far more sophisticated — most pre-trained models use subword tokenizers like BPE or WordPiece rather than simple character-level tokenization).
- **Deploy LLMs** once trained or fine-tuned.

### Beyond Text: Images and Audio

Hugging Face is not limited to working with just text; it also includes APIs to work with images and audio, covering tasks like image classification, object detection, speech-to-text, and text-to-speech.

### Comparing Hugging Face and OpenAI

You might be wondering how Hugging Face is different from OpenAI. Both provide models and allow us to build applications using them, but they are very different companies:

| | Hugging Face | OpenAI |
|---|---|---|
| **Model access** | Open-source; models can be downloaded and run locally or on servers you control | Closed-source; models are only accessible via OpenAI's API |
| **Where it runs** | On your own hardware (laptop, your own servers, or any cloud provider) | Only on servers OpenAI operates |
| **Cost** | Many models are free to use, though availability and usage rights depend on each model's individual license, and some hosted services/features are paid | Paid access required for advanced models, billed through their API |

### Why This Distinction Matters in Practice

This difference has real practical consequences beyond just licensing: running a Hugging Face model locally means your data never leaves your own infrastructure (relevant for privacy-sensitive applications), you have full control over costs (no per-request API billing, just your own compute), and you can inspect or modify the model's weights directly. The trade-off is that you're responsible for hosting, scaling, and maintaining the infrastructure yourself — whereas an API-based service like OpenAI's hands all of that operational burden to the provider, at the cost of ongoing usage fees and less control over the model itself.



## Using Hugging Face: The `pipeline` Function

When it comes to using Hugging Face models, we have two options:

1. **High-Level API**: This is accessible using the `pipeline` function and is user-friendly.
2. **Low-Level API**: This offers more flexibility and control over the model's usage.

Since this is a beginner-friendly lesson, we will focus just on using the high-level API, but keep in mind that Hugging Face API has much more to offer.

> **What's hiding behind the low-level API?** When you drop down a level, you're working directly with `AutoTokenizer` and `AutoModelForCausalLM` classes — which is much closer to what we built ourselves in `Learning-Creating-Transformer.ipynb`. The tokenizer there plays the same role as our `CharTokenizer`, and the model's `forward` pass plays the same role as our `DemoGPT.forward`. The `pipeline` function is essentially a convenience wrapper that bundles "tokenize → run model → decode output" into a single call, hiding the exact mechanics we implemented by hand.

### Text Generation with Hugging Face

To use a large language model with Hugging Face:

1. **Import the Pipeline Function**: Import the `pipeline` function from the Transformers package.
2. **Specify the Task**: Call the `pipeline` function and provide the name of the task, such as `"text-generation"`. This function returns another function that we can use to perform the selected task.
3. **Provide a Prompt**: For text generation, specify a prompt that will be used to generate the text.

To generate text using this prompt:

- Call the generator function returned by the `pipeline` call.
- Provide parameters such as the prompt, the maximum length of the output, and the number of sequences to generate.

```python
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
output = generator(
    "Once upon a time",
    max_length=30,
    num_return_sequences=2,
)
```

Notice the direct parallel to the `generate_with_prompt` function we wrote ourselves: `max_length` corresponds to how many tokens we loop and sample for, and `num_return_sequences` is just running that same generation loop multiple times with different random draws from `torch.multinomial`. The difference is that `gpt2` here is a real pre-trained model with billions of training tokens behind it, not our Tiny-Shakespeare-trained `DemoGPT`.

### Hugging Face's Versatility

Hugging Face supports a wide range of NLP-related tasks, including:

- **Text Generation**: As described above.
- **Question Answering**: Finding answers to questions based on a given context.
- **Text Summarization**: Summarizing large pieces of text.
- **Translation**: Translating text between different languages.
- **Text Classification**: Classifying text into categories.

In addition to NLP tasks, Hugging Face also supports:

- **Audio Tasks**: Such as audio classification and text-to-speech conversion.
- **Image and Video Tasks**: Including image classification, video classification, and image segmentation.

| Task | Pipeline name | Example use case |
|---|---|---|
| Text Generation | `text-generation` | Autocomplete, chatbots, story writing |
| Question Answering | `question-answering` | Search assistants, document Q&A |
| Summarization | `summarization` | Condensing articles or reports |
| Translation | `translation_xx_to_yy` | Language localization |
| Text Classification | `text-classification` | Spam detection, sentiment analysis |
| Audio Classification | `audio-classification` | Speaker/genre identification |
| Image Classification | `image-classification` | Object/scene recognition |

Every single one of these tasks, at its core, still reduces to the same building blocks we implemented: tokenize input into numeric ids, run them through stacked Transformer blocks (self-attention + feedforward), and interpret the final layer's output — the only thing that changes is what that output represents (next-token logits for generation, a start/end span for Q&A, a class label for classification, and so on).

### Example: Image Classification with Hugging Face

Here's how to perform image classification using Hugging Face:

1. **Specify the Task**: Use the `pipeline` function, specifying the task as `"image-classification"`.
2. **Pass an Image**: Provide an image file or an image URL for classification.
3. **Interpret the Output**: The output is a list with each element containing a label and an associated probability, indicating what the model thinks the image represents. For example, it might classify an image as likely being a cat.

```python
from transformers import pipeline

classifier = pipeline("image-classification")
result = classifier("https://example.com/cat.jpg")
# result -> [{"label": "tabby cat", "score": 0.93}, {"label": "tiger cat", "score": 0.04}, ...]
```

This should look familiar: that list of `(label, score)` pairs is conceptually the same as the probability distribution our `softmax` function produces over the vocabulary at each generation step — just applied over a fixed set of image classes instead of a fixed vocabulary of tokens. The model architecture underneath (often a Vision Transformer, or ViT) even reuses the same self-attention mechanism we built, just applied to patches of an image instead of tokens of text.


## More Hugging Face Pipelines in Action

#### Text Summarization

First, to use the high-level API from Hugging Face, we need to import the `pipeline` function from the `transformers` library. Let's see how we can summarize some text. To do this, we first need to create a pipeline and specify the summarization task. When we create it, this function will return a summarizer that we can then use to summarize our text.

```python
from transformers import pipeline

# Load the summarization pipeline
summarizer = pipeline("summarization")

# Text to be summarized
text = """
The Hub is home to over 5,000 datasets in more than 100 languages that can be used for a broad range of tasks across NLP, Computer Vision, and Audio. The Hub makes it simple to find, download, and upload datasets.
Datasets are accompanied by extensive documentation in the form of Dataset Cards and Dataset Preview to let you explore the data directly in your browser.
While many datasets are public, organizations and individuals can create private datasets to comply with licensing or privacy issues. You can learn more about Datasets here on Hugging Face Hub documentation.

The 🤗 datasets library allows you to programmatically interact with the datasets, so you can easily use datasets from the Hub in your projects.
With a single line of code, you can access the datasets; even if they are so large they don’t fit in your computer, you can use streaming to efficiently access the data.
"""

summary = summarizer(text, max_length=130, min_length=30)
print(summary[0]['summary_text'])
```

Under the hood, summarization models are usually **encoder-decoder** Transformers (like the T5/BART family we discussed earlier) rather than decoder-only ones like `DemoGPT`: the encoder reads the whole input text first and builds a contextual representation of it, and the decoder then generates the shorter summary token-by-token using cross-attention over that representation — the same cross-attention mechanism we compared against self-attention in the encoder-decoder section above. `max_length`/`min_length` simply bound how many tokens that decoding loop is allowed to produce, just like `max_new_tokens` would in our own `generate_with_prompt`.

#### Text Generation

Next, let's see how we can generate some text. The process here is very similar. We create a pipeline, provide a prompt that will be used as a starting point for the model to generate follow-up text, and we can also provide additional parameters like `max_length` and the number of sequences to return.

```python
# Load the text generation pipeline
generator = pipeline("text-generation")

# Define the prompt for the text generation
prompt = "Scientists from University of California, Berkeley has announced that"

# Generate text based on the prompt
generated_text = generator(prompt, max_length=100, num_return_sequences=1)
print(generated_text[0]['generated_text'])
```

This is the exact same decoder-only, autoregressive loop as our `generate_with_prompt`: the prompt is tokenized, fed through the stacked Transformer blocks, and at each step a next-token distribution is sampled from and appended to the sequence, repeating until `max_length` tokens have been produced.

#### Zero-Shot Classification

Another thing we will try is to classify text using so-called zero-shot classification. In this context, "shot" means how many examples we provide in a prompt for our model to understand what it should do. "Zero-shot" means we don't provide any example.

```python
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Text to classify
text = "Transformers support framework interoperability between PyTorch, TensorFlow, and JAX. This provides the flexibility to use a different framework at each stage of a model’s life; train a model in three lines of code in one framework, and load it for inference in another."

labels = ["technology", "biology", "space", "transport"]

results = classifier(text, candidate_labels=labels, hypothesis_template="This text is about {}.")
for label, score in zip(results['labels'], results['scores']):
  print(f"{label} -> {score}")
```

Zero-shot classification is a clever trick rather than a separate architecture: the model doesn't have a fixed set of output classes baked in. Instead, it turns each candidate label into a sentence (via `hypothesis_template`) and uses a model trained for **natural language inference** (does the premise entail the hypothesis?) to score how well the input text "entails" each label-sentence. The resulting `scores` are just another softmax-normalized probability distribution, this time over our arbitrary label set instead of a fixed vocabulary — the same mathematical object as our own `softmax` output, applied to a different set of categories each time we call it.

#### Image Captioning

Hugging Face is not limited to working just with text; we can also use it with images. For example, if we have an image, we can create a captioning model that will write a text caption for this image, describing what is in the image.

```python
from IPython.display import Image, display

# Replace the URL below with the URL of your image
image_url = 'https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png'

# Display the image in the notebook
display(Image(url=image_url))

captioner = pipeline(model="ydshieh/vit-gpt2-coco-en")

result = captioner(image_url)
print(result)
```

The model name is a clue to its architecture: `vit-gpt2` pairs a **Vision Transformer (ViT)** encoder (which turns image patches into embeddings the same way our token embedding layer turns token ids into vectors) with a **GPT-2** decoder (a decoder-only Transformer much like `DemoGPT`, just much larger and pre-trained on web-scale text) connected by cross-attention. In other words, captioning is image-to-text translation, built from the same encoder-decoder pattern as the French-translation example earlier, just with an image encoder instead of a text encoder.

| Pipeline | Architecture style | Closest concept we built |
|---|---|---|
| `summarization` | Encoder-decoder (T5/BART) | Cross-attention over our own self-attention |
| `text-generation` | Decoder-only (GPT-2) | Our `generate_with_prompt` loop, nearly 1:1 |
| `zero-shot-classification` | Encoder-only (BART-MNLI) | Our `softmax`, over custom labels instead of vocabulary |
| `image-captioning` | ViT encoder + GPT-2 decoder | Token embeddings (image patches) + our decoder loop |


## Controlling Text Generation: Model Parameters vs. Generation Parameters

To avoid some possible confusion, there are two types of parameters:

- **model parameters**, which are set during model training, and
- **generation parameters**, which can affect the result of text generation when we use a model after it is trained.

This distinction maps directly onto our own notebook: `embedding_dim`, `attention_heads`, `layers_num`, and friends in our `config` dict are **model parameters** — baked into the weights once training finishes, and unchangeable afterwards without retraining. Everything described below is a **generation parameter** — it only affects how we sample from the already-trained model's output distribution at inference time, exactly like the choices we made inside our own `generate`/`generate_with_prompt` functions.

#### Maximum Number of Tokens

We have already seen one parameter, which is the maximum number of tokens the text generation process should return. In Hugging Face, this is called the `max_length` parameter. As you can imagine, the higher this parameter is, the longer the output the text generation process will produce. However, one gotcha is that it defines a maximum number of tokens, and text generation can return an output that is shorter than this value. During the text generation process, a model can emit a special "end of sequence" token that will stop the generation process even before it reaches the specified limit.

> Our own `generate_with_prompt` doesn't have an end-of-sequence token or vocabulary entry for one — it always runs for a fixed, caller-specified number of steps — so this is one capability real pre-trained tokenizers add that our simple `CharTokenizer` does not.

#### Greedy Decoding

One simple option for generating text is to select the token with the highest probability every time a new token is generated. This approach, called "greedy decoding," picks the word with the highest probability and adds it to the input, repeating the process. However, this can lead to boring and repetitive results.

Greedy decoding is equivalent to swapping our `torch.multinomial(probabilities, num_samples=1)` call for `torch.argmax(probabilities)` — always taking the single most likely next token instead of sampling from the distribution.

#### Multinomial Sampling

To avoid repetitive results, we can introduce randomness by picking less probable tokens. This method, known as "multinomial sampling," uses the output of the softmax layer as a probability distribution to randomly select tokens. The higher the probability of a token, the more likely it will be selected. This approach provides a more varied and interesting output.

This is precisely what our `generate` and `generate_with_prompt` functions already do — we call `torch.multinomial` on the softmax output specifically to avoid the repetitive, deterministic output that greedy decoding produces.

#### Beam Search

Beam search is an algorithm used to generate multiple sequences at the same time, evaluating them to find the best result. Instead of generating a single sequence, beam search keeps "n" best sequences at each step, generating new sequences from these and repeating the process. This method helps in finding more optimal sequences by considering multiple possible outputs simultaneously.

Unlike greedy decoding or multinomial sampling — both of which only ever track one sequence — beam search tracks `num_beams` candidate sequences in parallel, extending all of them at every step and keeping only the overall highest-scoring subset. It trades extra compute for a better chance of finding a high-probability sequence overall, something neither of our own generation functions attempt (they only track a single running sequence).

#### Top-K Sampling

The `top_k` parameter specifies the number of tokens with the highest probability to consider when selecting the next token. For example, setting `top_k` to 3 means selecting the next token from the top three most probable tokens. Higher `top_k` values can lead to more creative outputs but may also produce less coherent text.

This is a middle ground between greedy decoding (`top_k=1`) and full multinomial sampling over the entire vocabulary (what our own code does, equivalent to `top_k` being the full vocabulary size): restrict the pool of candidates to the `top_k` highest-probability tokens, re-normalize their probabilities, and only then sample.

#### Top-P (Nucleus) Sampling

The `top_p` parameter specifies the cumulative probability of the most probable tokens to consider when selecting the next token. This method ensures that only tokens that collectively meet a certain probability threshold are considered, allowing for dynamic adjustment of the number of tokens based on their probabilities.

Where `top_k` fixes the *count* of candidate tokens, `top_p` fixes the *cumulative probability mass* instead — so the candidate pool can be small when the model is confident (probability concentrated on a few tokens) and large when it's uncertain (probability spread thin across many tokens).

#### Temperature

The `temperature` parameter adjusts the probability distribution returned by the softmax layer. Lowering the temperature increases the chances of selecting tokens with higher logits, making the output more deterministic. Increasing the temperature flattens the probability distribution, allowing for more diverse token selection.

Mechanically, temperature `T` divides the logits before the softmax call: `softmax(logits / T)`. Our own `softmax` implementation (and the `F.softmax` calls in `generate`) always effectively used `T = 1`. As `T → 0`, the distribution sharpens toward a single token (approaching greedy decoding); as `T` increases beyond 1, the distribution flattens toward uniform, making even low-probability tokens more likely to be sampled.

### Using Parameters with Hugging Face

In Hugging Face, these parameters can be set when using the generator function. For example, `do_sample` enables multinomial sampling, and `num_beams` specifies the number of sequences in beam search. Parameters like `top_k`, `top_p`, and `temperature` control the randomness and creativity of the generated output, allowing for fine-tuning the behavior of the language model according to specific requirements.

```python
generated_text = generator(
    prompt,
    max_length=100,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    num_return_sequences=1,
)
```

| Parameter | Hugging Face name | What our own code does |
|---|---|---|
| Greedy decoding | `do_sample=False` (default) | Not implemented — we always sample |
| Multinomial sampling | `do_sample=True` | `torch.multinomial` on the full softmax output |
| Beam search | `num_beams=n` | Not implemented — we only track one sequence |
| Top-k | `top_k=n` | Not implemented — we sample over the whole vocabulary |
| Top-p / nucleus | `top_p=p` | Not implemented — same as above |
| Temperature | `temperature=t` | Not implemented — equivalent to always using `T=1` |
